# Hoshen–Kopelman: build and test the code

I develop this teaching implementation one part at a time. Run the cells in order. Each new part has a small check before I use it in the next part. This is a new worked implementation, not a reconstructed record of my historical editing sequence.

Four-edge adjacency in 2D; bounded or periodic boundaries. Requirements: Python, NumPy and Jupyter.

References: [Hoshen and Kopelman (1976)](https://doi.org/10.1103/PhysRevB.14.3438), [Tobin Fricke’s explanation](https://www.ocf.berkeley.edu/~fricke/projects/hoshenkopelman/hoshenkopelman.html).

## 1. Write a small grid and inspect it

I begin with a small array that I can inspect by eye. A value of 1 identifies an occupied cell; 0 identifies background. Rows run downward and columns run to the right. These values and the labels are dimensionless. I count connections through shared edges, so corner contact alone does not connect two cells.

In [1]:
import numpy as np

mask = np.array([[1, 0, 1],
                 [1, 1, 1],
                 [0, 0, 0]], dtype=bool)
assert mask.shape == (3, 3)
assert mask.sum() == 5
print(mask.astype(int))


[[1 0 1]
 [1 1 1]
 [0 0 0]]


## 2. Find the representative of a label

The first row appears to contain two separate clusters. Later cells connect them. I therefore need a way to record that two labels belong to the same cluster. In the parent list, parent[k] gives the next label to follow. A label that points to itself is the representative. Label 0 is reserved for background. I first test lookup without scanning a grid.

In [2]:
def find_label(parent, label):
    while parent[label] != label:
        label = parent[label]
    return label

parent = [0, 1, 1, 2]
assert find_label(parent, 3) == 1
assert find_label(parent, 1) == 1
print("Label 3 belongs to the cluster represented by label 1.")


Label 3 belongs to the cluster represented by label 1.


## 3. Merge two labels

I find the representatives before merging labels. Otherwise I could link the wrong entries or form a cycle in the parent list. I make the larger representative point to the smaller one. This gives predictable labels in the worked example. I do not need a more elaborate data structure for this small lesson.

In [3]:
def merge_labels(parent, first, second):
    first_root = find_label(parent, first)
    second_root = find_label(parent, second)
    root = min(first_root, second_root)
    parent[max(first_root, second_root)] = root
    return root

parent = [0, 1, 2, 3]
merge_labels(parent, 2, 3)
merge_labels(parent, 1, 3)
assert [find_label(parent, k) for k in (1, 2, 3)] == [1, 1, 1]
print("Three labels now identify one cluster.")


Three labels now identify one cluster.


## 4. Scan the grid once

I visit rows from top to bottom, and cells within each row from left to right. Only the upper and left neighbors have already been visited. If both are background, I create a label. If one is occupied, I use its label. If both are occupied, I merge their labels. Each shared-edge connection is considered when its later cell is visited. The returned labels are provisional: some different numbers may already refer to the same cluster.

In [4]:
def scan_grid(mask):
    rows, columns = mask.shape
    labels = np.zeros(mask.shape, dtype=int)
    parent = [0]
    for row in range(rows):
        for column in range(columns):
            if not mask[row, column]:
                continue
            above = labels[row - 1, column] if row > 0 else 0
            left = labels[row, column - 1] if column > 0 else 0
            if above == 0 and left == 0:
                new_label = len(parent)
                parent.append(new_label)
                labels[row, column] = new_label
            elif above == 0:
                labels[row, column] = find_label(parent, left)
            elif left == 0:
                labels[row, column] = find_label(parent, above)
            else:
                labels[row, column] = merge_labels(parent, above, left)
    return labels, parent

labels, parent = scan_grid(mask)
assert labels[0, 0] != labels[0, 2]
assert find_label(parent, labels[0, 0]) == find_label(parent, labels[0, 2])
print("Provisional labels:")
print(labels)


Provisional labels:
[[1 0 2]
 [1 1 1]
 [0 0 0]]


## 5. Replace provisional labels and count cells

I now replace each occupied cell label by its representative. I renumber the surviving representatives from 1 upward. The size of a cluster is the number of cells with its final label. For a uniform two-dimensional grid, its area is this count multiplied by the cell area. A count has no units; an area has the units of the squared grid spacing. The largest-cluster fraction is the largest count divided by the occupied-cell count, not the total grid-cell count.

In [5]:
def finish_labels(labels, parent):
    final = np.zeros(labels.shape, dtype=int)
    numbers = {}
    for row in range(labels.shape[0]):
        for column in range(labels.shape[1]):
            label = labels[row, column]
            if label == 0:
                continue
            root = find_label(parent, label)
            if root not in numbers:
                numbers[root] = len(numbers) + 1
            final[row, column] = numbers[root]
    sizes = [int(np.count_nonzero(final == k))
             for k in range(1, len(numbers) + 1)]
    return final, sizes

final, sizes = finish_labels(labels, parent)
assert sizes == [5]
assert np.array_equal(final > 0, mask)
print("Final labels:")
print(final)
print("Cluster sizes:", sizes)


Final labels:
[[1 0 1]
 [1 1 1]
 [0 0 0]]
Cluster sizes: [5]


## 6. Add periodic connections as a separate step

I keep the bounded scan unchanged. For periodic data I then connect occupied cells at matching positions on opposite edges. The top cell in a column is adjacent to the bottom cell in that same column. The left and right cells in a row are adjacent in the same way. I merge these labels before producing the final counts. The two-cell example has two bounded clusters and one periodic cluster. It is a small island crossing the displayed seam; it does not form a loop around the whole domain.

In [6]:
def join_periodic_edges(labels, parent):
    rows, columns = labels.shape
    for column in range(columns):
        top = labels[0, column]
        bottom = labels[rows - 1, column]
        if top and bottom:
            merge_labels(parent, top, bottom)
    for row in range(rows):
        left = labels[row, 0]
        right = labels[row, columns - 1]
        if left and right:
            merge_labels(parent, left, right)


def hoshen_kopelman(mask, periodic=False):
    mask = np.asarray(mask, dtype=bool)
    if mask.ndim != 2 or min(mask.shape) == 0:
        raise ValueError("Use a nonempty two-dimensional grid.")
    labels, parent = scan_grid(mask)
    if periodic:
        join_periodic_edges(labels, parent)
    return finish_labels(labels, parent)

seam = np.zeros((5, 5), dtype=bool)
seam[0, 2] = seam[-1, 2] = True
assert hoshen_kopelman(seam)[1] == [1, 1]
assert hoshen_kopelman(seam, periodic=True)[1] == [2]
print("Seam island: 2 bounded clusters; 1 periodic cluster.")


Seam island: 2 bounded clusters; 1 periodic cluster.


## 7. Test examples before analyzing a microstructure

I test an empty grid, a filled grid, a corner contact and a ring. A ring and a filled square each have one connected component. Cluster labeling cannot distinguish their hole counts. I also test a rectangular grid so that accidentally exchanging the row and column dimensions cannot remain hidden.

In [7]:
assert hoshen_kopelman(np.zeros((3, 5), bool))[1] == []
assert hoshen_kopelman(np.ones((3, 5), bool))[1] == [15]
assert hoshen_kopelman(np.eye(3, dtype=bool))[1] == [1, 1, 1]
ring = np.ones((5, 5), dtype=bool)
ring[1:4, 1:4] = False
assert hoshen_kopelman(ring)[1] == [16]
rectangle = np.zeros((3, 5), dtype=bool)
rectangle[1, 0] = rectangle[1, -1] = True
assert hoshen_kopelman(rectangle)[1] == [1, 1]
assert hoshen_kopelman(rectangle, periodic=True)[1] == [2]
print("All small-grid checks passed.")


All small-grid checks passed.


## 8. Apply a threshold and analyze both phases

The composition c and threshold are dimensionless here. I define the high-composition phase using c >= 0.5 and the other phase using c < 0.5. I measure each separately. The occupied fraction uses all grid cells in its denominator. For an absent phase I report a largest-cluster fraction of zero by convention. In a real simulation I would load the saved field instead of this small example. I do not infer winding, holes or numerical convergence from these counts.

In [8]:
composition = np.array([[0.8, 0.2, 0.7],
                        [0.9, 0.6, 0.8],
                        [0.1, 0.2, 0.1]])
for name, phase in [("High composition", composition >= 0.5),
                    ("Low composition", composition < 0.5)]:
    labels, sizes = hoshen_kopelman(phase, periodic=True)
    occupied = int(phase.sum())
    largest_fraction = max(sizes) / occupied if occupied else 0.0
    assert sum(sizes) == occupied
    print(name, "occupied fraction:", occupied / phase.size,
          "clusters:", len(sizes), "largest-cluster fraction:", largest_fraction)


High composition occupied fraction: 0.5555555555555556 clusters: 1 largest-cluster fraction: 1.0
Low composition occupied fraction: 0.4444444444444444 clusters: 1 largest-cluster fraction: 1.0


## What these checks establish

The labels and counts describe shared-edge connectivity of the specified binary grid. They do not prove a periodic winding loop or determine the number of holes. I use displacement-aware connectivity for winding and a filled-region topology calculation for holes. Continue with the [topology tutorial](https://saswataiith.github.io/files/mesoscale/examples/topology-connectivity.html). For 3D, use six face-sharing neighbors and add opposite-face connections along all three axes.

The small teaching implementation favors clear steps over performance. Its label lookup has no path compression. For large fields use the existing full-volume implementation linked from the tutorial.